# Session 8 — Google Earth Engine II: large-scale extraction (HRSL population)
*Séance 8 — Google Earth Engine II : extraction à grande échelle (population HRSL)*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/08_gee_hrsl_population.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | — not in track / hors parcours |

**By the end of this session you will be able to:**

1. Use community datasets (awesome-gee-community-catalog) in Earth Engine.
2. Compute zonal sums at the right scale and derive population densities.
3. Compare administrative units in two countries and export results.
4. Prepare for the midterm practical exam.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

### Connect to Google Earth Engine
Since 13 November 2024, Earth Engine requires a **Google Cloud project** registered for Earth Engine (free for noncommercial use).
One-time setup: go to <https://code.earthengine.google.com/register>, choose *Unpaid usage → Academia & Research*, and create a project
(e.g. `ee-yourname`). Then write its ID below.

In [ ]:
import ee, geemap

EE_PROJECT = "ee-yourname"   # <-- replace with YOUR Cloud project ID

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()          # opens a sign-in link the first time
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready:", ee.String("Hello from Earth Engine").getInfo())

## Part A — Lecture

The **High Resolution Settlement Layer (HRSL)** (Meta & CIESIN) estimates population at **~30 m** by combining census counts with
buildings detected by deep learning on satellite imagery. It is available in Earth Engine through the community catalog
(`projects/sat-io/open-datasets/hrsl/...`) as tiles in an ImageCollection; layers exist for the general population and
for subgroups (men, women, youth 15–24, elderly 60+, children under 5, women of reproductive age).

In [ ]:
import geopandas as gpd

WARD_FILES = {"ZA1011001": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp",   # Western Cape
              "ZA6372040": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA6372040.shp"}   # North West


def ward_fc(code):
    """Upload a ward polygon from the course repository to Earth Engine as a FeatureCollection."""
    return geemap.geopandas_to_ee(gpd.read_file(WARD_FILES[code])[["ADM4_PCODE", "geometry"]])


ward = ward_fc("ZA1011001")
print("Ward area (km²):", ward.geometry().area(maxError=1).divide(1e6).getInfo())

In [ ]:
HRSL = {name: ee.ImageCollection(f"projects/sat-io/open-datasets/hrsl/{asset}").mosaic()
        for name, asset in {"total": "hrslpop", "men": "hrsl_men", "women": "hrsl_women",
                            "youth_15_24": "hrsl_youth", "elderly_60_plus": "hrsl_elderly_over_sixty",
                            "children_under_5": "hrsl_children_under_five"}.items()}

Map = geemap.Map()
Map.centerObject(ward, 11)
Map.addLayer(HRSL["total"].clip(ward), {"min": 0, "max": 10, "palette": ["yellow", "orange", "red"]}, "HRSL population")
Map.addLayer(ward.style(color="black", fillColor="00000000"), {}, "Ward")
Map

### Zonal sums
Population is a **count per pixel**, so we use `Reducer.sum()` at the native scale (30 m). Using `mean()` (as in the 2022
version) gives people per pixel, not people per ward.

In [ ]:
def population(fc, layer="total"):
    stats = HRSL[layer].reduceRegion(reducer=ee.Reducer.sum(), geometry=fc.geometry(), scale=30, maxPixels=1e10)
    return ee.Number(stats.values().get(0))


total = population(ward).getInfo()
area_km2 = ward.geometry().area(maxError=1).divide(1e6).getInfo()
print(f"Population ≈ {total:,.0f} people, density ≈ {total / area_km2:,.1f} people/km²")

## Part B — Lab

**Task 1 — Demographic profile.** For the ward, build a dict `profile` with the population of each HRSL layer (keys of
`HRSL`) and the share of women `share_women` (women / total).

In [ ]:
# YOUR CODE HERE
profile = None
share_women = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(set(profile) == set(HRSL))
check(0.4 < share_women < 0.6, "The share of women is usually close to 50%")

**Task 2 — Another country.** Using FAO GAUL (`FAO/GAUL/2015/level2`), select the **Menoua** department of Cameroon
(`ADM0_NAME == 'Cameroon'`, `ADM2_NAME == 'Menoua'`) into `menoua`, then compute its population `menoua_pop`.

In [ ]:
# YOUR CODE HERE
menoua = None
menoua_pop = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(menoua_pop > 50_000, "Check your filter: Menoua has several hundred thousand inhabitants")

**Task 3 — Export.** Create a FeatureCollection with one feature per unit (ward, Menoua) holding `name`, `population`
and `density_km2`, and start an export task to Google Drive named `hrsl_population` (CSV).

In [ ]:
# YOUR CODE HERE
task = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(task.status()["state"] in ("READY", "RUNNING", "COMPLETED"))

### Midterm practical exam (next part of this session)
Individual, 90 minutes, open notes, **no generative AI**. You will receive a new polygon and must: (1) load and reproject
it, (2) compute an index or a rainfall/population statistic with Earth Engine, (3) produce a clean figure and a 5-line
interpretation. Graded on correctness (50%), code clarity (25%) and interpretation (25%).

**❓ Question —** HRSL depends on building detection. In which kinds of settlements could it under- or over-estimate population? Why does this matter for planning?

*Your answer here.*

## Summary

- Community catalogs extend Earth Engine with datasets such as HRSL.
- Counts → `Reducer.sum()` at native scale; intensities → `mean()`.
- The same function scales to wards, departments or countries.

**To go further:**

- Tiecke, T. G. et al. (2017). Mapping the world population one building at a time. arXiv:1712.05839.
- awesome-gee-community-catalog: https://gee-community-catalog.org

---
*Adapted from the 2022 ML4EO bootcamp HRSL notebooks (N. C. Fouotsa Manfouo).*